In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    String payload = """
        exec('''%s''')
        print('%s')
        """.formatted(code.replace("'", "\\'"), marker);

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}


In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


In [ ]:
%%jshell

runPy("""
# Download the dataset file to local storage (from publicly shared drive file)
# Original source: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
# (The rest of the notebook expects the original housing.csv to be present in the current directory)
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP');
""");


In [ ]:
%%jshell

runPy("""
# Load the dataset
import pandas as pd
DATASET_PATH = str(NOTEBOOK_DIR / "housing.csv")
df = pd.read_csv(DATASET_PATH)
""");


In [ ]:
%%jshell

runPy("""
# Display all the number of nan values in each column and type
column_stats_raw = []
for column_name in df.columns:
  column_stats_raw.append((column_name, df[column_name].dtype, df[column_name].isna().sum()))
column_stats_df = pd.DataFrame(column_stats_raw, columns=['Column name', 'Type', 'Number of null values'])

print(column_stats_df.to_string())
""");


In [ ]:
%%jshell

runPy("""
# Deletion of rows with missing data

# Scenario: We are trying to test the correlation between LotFrontage and SalePrice,
# therefore we remove the rows that have NA values for them
df.dropna(subset=["LotFrontage", "SalePrice"])
""");


In [ ]:
%%jshell

runPy("""
# Mean/Median Imputation

# We are considering LotFrontage again as it is in float (numerical) and has null values

meanLotFrontage = df["LotFrontage"].mean()
medianLotFrontage = df["LotFrontage"].median()

meanImputedLotFrontage = df["LotFrontage"].fillna(meanLotFrontage)
medianImputedLotFrontage = df["LotFrontage"].fillna(medianLotFrontage)

# Showing all values might not be the best option here given the size,
# therefore using mean/median to show that math is maintained

print("Mean before performing mean imputation:", meanLotFrontage)
print("Mean after performing mean imputation:", meanImputedLotFrontage.mean())
print()
print("Median before performing median imputation:", medianLotFrontage)
print("Median after performing median imputation:", medianImputedLotFrontage.median())
print()
print("Both are almost the same/maintained because of the kind of imputation performed here.")
""");


In [ ]:
%%jshell

runPy("""
# Mode imputation
# (Can be considered in cases of non-numeric values where mean/median averaging isn't possible)

print("Unique values of BsmtQual/Basement Quality column:", set(df["BsmtQual"]))

# Multiple modes can exist if max count is same, therefore considering only the first one for imputation
modeBsmtQual = df["BsmtQual"].mode()[0]

print("Most repeated value of BsmtQual column:", modeBsmtQual)

print()
print("Current number of NaN values in BsmtQual:", df["BsmtQual"].isna().sum())
modeImputedBsmtQual = df["BsmtQual"].fillna(modeBsmtQual)
print("After imputation, the number of NaN values now are,", modeImputedBsmtQual.isna().sum())
""");


In [ ]:
%%jshell

runPy("""
# Arbitrary value imputation (remaining)
# (Value chosen by analyst with his domain knowledge or their understanding of the dataset)

# For example, we could choose

arbitary_value = 21.0

print("Mean of LotFrontage before arbitrary value imputation:", df["LotFrontage"].mean())
arbitaryValueImputedCol = df["LotFrontage"].fillna(arbitary_value)
print("Mean of LotFrontage after arbitrary value imputation: ", arbitaryValueImputedCol.mean())
""");


In [ ]:
%%jshell

runPy("""
# End of tail Imputation
minLotFrontage = df["LotFrontage"].min()
maxLotFrontage = df["LotFrontage"].max()

chosenTail = minLotFrontage

print("Mean before end of tail imputation:", df["LotFrontage"].mean())
tailImputedLotFrontage = df["LotFrontage"].fillna(chosenTail)
print("Mean after end of tail imputation:", tailImputedLotFrontage.mean())
""");


In [ ]:
%%jshell

runPy("""
# Random sample imputation (runtime error with logic)
import random

print("NaN values in FireplaceQu before sampling imputation:", df["FireplaceQu"].isna().sum())

# Random sample Imputation (on Fireplace Quality column)
random_sample_size = 10
random_sample = df["FireplaceQu"].dropna().sample(n=random_sample_size)

impute_random_sample = lambda x: random_sample.sample().iloc[0] if pd.isnull(x) else x

# Apply the lambda function to impute missing values in the "FireplaceQu" column
random_sample_imputed_col = df["FireplaceQu"].apply(impute_random_sample)

print("NaN values in FireplaceQu after sampling imputation: ", random_sample_imputed_col.isna().sum())
""");


In [ ]:
%%jshell

runPy("""
#	Frequent category Imputation (same as mode imputation)
print("Unique values of GarageType column:", set(df["GarageType"]))

# Multiple modes can exist if max count is same, therefore considering only the first one for imputation
modeGarageType = df["GarageType"].mode()[0]

print("Most repeated value of GarageType column:", modeGarageType)

print()
print("Current number of NaN values in BsmtQual:", df["GarageType"].isna().sum())
freqCatImputatedCol = df["GarageType"].fillna(modeGarageType)
print("After imputation, the number of NaN values now are:", freqCatImputatedCol.isna().sum())
""");


In [ ]:
%%jshell

runPy("""
# Adding a new category as "missing"
# (Replacing NaN values with explicit category such as "missing" as a common category for them)

missing_category = "Unknown"

print("Set of unique values in GarageType col before given imputation:", set(df["GarageType"]))
imputedColumn = df["GarageType"].fillna(missing_category)
print("Set of unique values in GarageType col after given imputation:", set(imputedColumn))
""");


In [ ]:
%%jshell

runPy("""
# Random sample imputation (done earlier/repetition)
""");


In [ ]:
%%jshell

runPy("""
# Regression
# We'll be using SalePrice (x) to predict LotFrontage (y) for missing/nan values
from sklearn.linear_model import LinearRegression

model = LinearRegression()
imputed_df = df.dropna(subset=["LotFrontage"])[["LotFrontage", "SalePrice"]]
X = imputed_df.SalePrice.values.reshape(len(imputed_df), 1)
Y = imputed_df.LotFrontage.values.reshape(len(imputed_df), 1)
model.fit(X, Y)

regression_imputed_lot_frontage = df[["LotFrontage", "SalePrice"]].apply(lambda x: model.predict([[x["SalePrice"]]])[0][0] if pd.isnull(x["LotFrontage"]) else x["LotFrontage"], axis=1)

print("Mean before regression imputation of LotFrontage:", df["LotFrontage"].mean())
print("Number NA values before imputation:", df["LotFrontage"].isna().sum())
print()
print("Mean after regression imputation of LotFrontage:", regression_imputed_lot_frontage.mean())
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
